# Training runner: LoRA fine-tuning (C3)

Thin runner; logic lives in `src/spider_sql/training.py`. Runtime: GPU T4. Checkpoints go to Drive:
after a disconnect, rerun the setup cells and the same training cell, and it resumes.

Order (SPEC 6 and 11): smoke test, then the 4-run tuning grid on 0.5B with seed 0, scored on
validation only. The winning config is frozen in SPEC and docs/DECISIONS.md before any seed runs.

In [ ]:
from google.colab import drive

MOUNT = "/content/drive"  # the only path to edit if you mount elsewhere
drive.mount(MOUNT)

In [ ]:
!test -d /content/repo || git clone https://github.com/frangarciaproff/spider-sql-lora-ladder.git /content/repo
!git -C /content/repo pull --ff-only
%cd /content/repo
!pip install -q -e . peft
!pip install -q unsloth
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

DRIVE = f"{MOUNT}/MyDrive/fine-tuning"
# writing to an unmounted path silently uses the runtime disk, which is lost
assert os.path.ismount(MOUNT), f"{MOUNT} is not a mounted Drive"
assert os.path.isfile(f"{DRIVE}/spider_bundle.zip"), f"no bundle in {DRIVE}: wrong account?"
BUNDLE_SHA256 = "aacfdd35b1a6afc369ef028cff59c39ac8639fd813c3b092fb7e5a1615e2b44f"
!python scripts/get_data.py --zip {DRIVE}/spider_bundle.zip --dest /content/data --sha256 {BUNDLE_SHA256}
os.environ["SPIDER_DIR"] = "/content/data/spider"
os.environ["TESTSUITE_DIR"] = "/content/data/testsuite"

In [ ]:
import torch

assert torch.cuda.is_available(), (
    f"No usable GPU (torch {torch.__version__}, CUDA {torch.version.cuda}). "
    "Runtime > Change runtime type > T4; if T4 is refused, the free quota is used up."
)
print(torch.cuda.get_device_name(0))

MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
TAG = "qwen05b"
PER_DEVICE, ACCUM = 8, 2  # effective batch 16 (SPEC 6); use 4, 4 for 1.5B
BACKEND = "unsloth"  # switch to "peft" if unsloth fails to install or import
ADAPTERS = f"{DRIVE}/adapters"
PRED = f"{DRIVE}/predictions"
RES = f"{DRIVE}/results"

## 1. Smoke test: 30 optimizer steps on 480 examples (never reported)
Check: the printed token-length stats, a falling loss, peak VRAM, seconds per step.
If unsloth errors, set `BACKEND = "peft"` and rerun this cell.

In [ ]:
!python scripts/train.py --model {MODEL} --backend {BACKEND} --per-device-batch {PER_DEVICE} --grad-accum {ACCUM} --limit 480 --max-steps 30 --out-dir {ADAPTERS}/smoke_{TAG}_{BACKEND}
!python -c "import json; h = json.load(open('{ADAPTERS}/smoke_{TAG}_{BACKEND}/log_history.json')); print([round(x['loss'], 3) for x in h if 'loss' in x])"

## 2. Tuning grid (0.5B, seed 0): lr in {1e-4, 2e-4} x epochs in {2, 3}
Each run trains, then generates and scores on validation. One run per cell, so a disconnect
costs at most one run; rerunning a cell resumes it.

In [ ]:
def run_grid_point(lr, epochs, seed=0):
    run = f"{TAG}_c3_lr{lr:g}_ep{epochs}_seed{seed}"
    !python scripts/train.py --model {MODEL} --backend {BACKEND} --per-device-batch {PER_DEVICE} --grad-accum {ACCUM} --lr {lr} --epochs {epochs} --seed {seed} --out-dir {ADAPTERS}/{run}
    if _exit_code != 0:  # noqa: F821  (set by IPython after each ! command)
        raise RuntimeError(f"step failed for {run}; see the output above")
    !python scripts/generate.py --model {MODEL} --condition adapter --adapter {ADAPTERS}/{run}/adapter --split validation --batch-size 32 --out-dir {PRED}/{run}
    if _exit_code != 0:  # noqa: F821  (set by IPython after each ! command)
        raise RuntimeError(f"step failed for {run}; see the output above")
    !python scripts/score_predictions.py --pred-dir {PRED}/{run} --split validation --out-dir {RES}/{run}

In [ ]:
run_grid_point(1e-4, 2)

In [ ]:
run_grid_point(2e-4, 2)

In [ ]:
run_grid_point(1e-4, 3)

In [ ]:
run_grid_point(2e-4, 3)

## 3. Final seeds with the frozen config (SPEC 6: lr 2e-4, 2 epochs)
One cell per seed. Each trains (or resumes), then generates and scores validation and dev.
For 0.5B, seed 0 is the grid run: its cell only adds the dev evaluation.
For 1.5B set MODEL, TAG, PER_DEVICE/ACCUM = 4, 4 and GEN_BATCH = 16 in the configuration cell first.

In [ ]:
LR, EPOCHS = 2e-4, 2  # frozen 2026-10-18
GEN_BATCH = 32  # 16 for 1.5B


def check(step):
    if _exit_code != 0:  # noqa: F821  (set by IPython after each ! command)
        raise RuntimeError(f"{step} failed; see the output above")


def run_seed(seed):
    run = f"{TAG}_c3_lr{LR:g}_ep{EPOCHS}_seed{seed}"
    !python scripts/train.py --model {MODEL} --backend {BACKEND} --per-device-batch {PER_DEVICE} --grad-accum {ACCUM} --lr {LR} --epochs {EPOCHS} --seed {seed} --out-dir {ADAPTERS}/{run}
    check(f"training {run}")
    for split in ["validation", "dev"]:
        !python scripts/generate.py --model {MODEL} --condition adapter --adapter {ADAPTERS}/{run}/adapter --split {split} --batch-size {GEN_BATCH} --out-dir {PRED}/{run}
        check(f"generation {run} {split}")
        !python scripts/score_predictions.py --pred-dir {PRED}/{run} --split {split} --out-dir {RES}/{run}
        check(f"scoring {run} {split}")
    !ls -la {ADAPTERS}/{run} {RES}/{run}

In [ ]:
run_seed(0)

In [ ]:
run_seed(1)

In [ ]:
run_seed(2)

## 4. Copy results
Download `results/<run>/` and `adapters/<run>/train_meta.json` + `log_history.json` into the repo
(adapter weights stay on Drive). The test split is not run here: it waits for review point 3.